## 1. 检查环境

In [22]:
# 检查环境
import sys
import os

print(f"Python: {sys.version.split()[0]} | 路径: {sys.executable}")

try:
    import pandas as pd
    import numpy as np
    print(f"✅ pandas {pd.__version__}, numpy {np.__version__}")
except ImportError as e:
    print(f"❌ 依赖未安装: {e}")
    print("\n" + "="*60)
    print("📦 安装依赖包（请在终端运行）:")
    print("="*60)
    print("cd /Users/zhanglifeng12703/Documents/OverseasPython/Mytest")
    print("source .venv/bin/activate")
    print("pip install pandas numpy")
    print("\n如果遇到 SSL 错误，使用:")
    print("pip install --trusted-host pypi.org --trusted-host pypi.python.org --trusted-host files.pythonhosted.org pandas numpy")
    print("="*60)
    print("\n安装完成后，重启 Cursor 并重新运行此代码块")
    raise

Python: 3.12.12 | 路径: /Users/zhanglifeng12703/Documents/OverseasPython/.venv/bin/python
✅ pandas 3.0.0, numpy 2.4.1


## 2. 导入库

In [23]:
import pickle
import pandas as pd
import numpy as np
import os
import json
from pprint import pprint

## 3. 设置文件路径

In [24]:
# 设置文件路径
base_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'Mytest':
    current = os.getcwd()
    while current != os.path.dirname(current):
        if os.path.basename(current) == 'Mytest':
            base_dir = current
            break
        current = os.path.dirname(current)

pkl_file_path = os.path.join(base_dir, 'cdc_pickle_pass_fpd7.pkl')
output_csv_path = os.path.join(base_dir, 'cdc_pickle_pass_fpd7.csv')

print(f"PKL文件: {pkl_file_path}")
print(f"输出CSV: {output_csv_path}")
print(f"文件存在: {os.path.exists(pkl_file_path)}")
pkl_file_path

PKL文件: /Users/zhanglifeng12703/Documents/OverseasPython/Mytest/cdc_pickle_pass_fpd7.pkl
输出CSV: /Users/zhanglifeng12703/Documents/OverseasPython/Mytest/cdc_pickle_pass_fpd7.csv
文件存在: True


'/Users/zhanglifeng12703/Documents/OverseasPython/Mytest/cdc_pickle_pass_fpd7.pkl'

## 9. 分析 response_body 中的特征

In [25]:
# 分析 response_body 中的特征
def flatten_dict(d, parent_key='', sep='.'):
    """打平嵌套字典，将嵌套的键用点号连接"""
    if d is None:
        return {}
    
    if not isinstance(d, dict):
        return {parent_key: d} if parent_key else {}
    
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        
        if isinstance(v, dict) and v:
            flattened = flatten_dict(v, new_key, sep)
            items.extend(flattened.items())
        elif isinstance(v, list):
            if len(v) == 0:
                items.append((new_key, None))
            elif isinstance(v[0], dict):
                all_keys = set()
                for item in v:
                    if isinstance(item, dict):
                        flat_item = flatten_dict(item, '', sep)
                        all_keys.update(flat_item.keys())
                
                for key in all_keys:
                    values = []
                    for item in v:
                        if isinstance(item, dict):
                            flat_item = flatten_dict(item, '', sep)
                            if key in flat_item:
                                val = flat_item[key]
                                if val is not None:
                                    values.append(str(val))
                    
                    if values:
                        unique_values = list(dict.fromkeys(values))
                        if len(unique_values) == 1:
                            items.append((f"{new_key}{sep}{key}", unique_values[0]))
                        else:
                            items.append((f"{new_key}{sep}{key}", '|'.join(unique_values)))
                    else:
                        items.append((f"{new_key}{sep}{key}", None))
            else:
                non_none_values = [str(item) for item in v if item is not None]
                if non_none_values:
                    items.append((new_key, '|'.join(non_none_values)))
                else:
                    items.append((new_key, None))
        else:
            items.append((new_key, v))
    
    return dict(items)

print("=" * 70)
print("特征分析")
print("=" * 70)

# 收集所有特征
all_features = {}
feature_stats = {}
sample_count = min(100, len(df))  # 分析前100条数据作为样本

print(f"\n正在分析前 {sample_count} 条数据的 response_body...")

for idx in range(sample_count):
    if idx % 20 == 0:
        print(f"  进度: {idx}/{sample_count}")
    
    response_body = df.iloc[idx]['response_body']
    
    if pd.isna(response_body) or response_body is None or response_body == '':
        continue
    
    try:
        if isinstance(response_body, str):
            json_obj = json.loads(response_body)
        else:
            json_obj = response_body
        
        flattened = flatten_dict(json_obj)
        
        # 统计特征
        for key, value in flattened.items():
            if key not in all_features:
                all_features[key] = []
                feature_stats[key] = {'count': 0, 'null_count': 0, 'sample_values': []}
            
            all_features[key].append(value)
            feature_stats[key]['count'] += 1
            
            if value is None or (isinstance(value, str) and value.strip() == ''):
                feature_stats[key]['null_count'] += 1
            elif len(feature_stats[key]['sample_values']) < 3:
                feature_stats[key]['sample_values'].append(str(value)[:50])
    
    except json.JSONDecodeError as e:
        continue
    except Exception as e:
        continue

print(f"\n✅ 分析完成！")
print(f"📊 共发现 {len(all_features)} 个特征字段")

# 显示特征统计
print(f"\n{'='*70}")
print("特征统计信息（前50个）")
print(f"{'='*70}")

sorted_features = sorted(feature_stats.items(), key=lambda x: x[1]['count'], reverse=True)

print(f"\n{'序号':<5} {'特征名':<50} {'出现次数':<10} {'缺失率':<10} {'示例值'}")
print("-" * 100)

for i, (feature_name, stats) in enumerate(sorted_features[:50], 1):
    null_rate = (stats['null_count'] / stats['count']) * 100 if stats['count'] > 0 else 0
    sample = stats['sample_values'][0] if stats['sample_values'] else 'N/A'
    if len(sample) > 30:
        sample = sample[:27] + '...'
    
    print(f"{i:<5} {feature_name:<50} {stats['count']:<10} {null_rate:>6.1f}%   {sample}")

if len(sorted_features) > 50:
    print(f"\n... 还有 {len(sorted_features) - 50} 个特征未显示")

print(f"\n{'='*70}")

特征分析

正在分析前 100 条数据的 response_body...
  进度: 0/100
  进度: 20/100
  进度: 40/100
  进度: 60/100
  进度: 80/100

✅ 分析完成！
📊 共发现 82 个特征字段

特征统计信息（前50个）

序号    特征名                                                出现次数       缺失率        示例值
----------------------------------------------------------------------------------------------------
1     claveOtorgante                                     100           0.0%   004676
2     consultas.fechaConsulta                            100           0.0%   2025-11-25|2025-11-04|2025-...
3     consultas.claveUnidadMonetaria                     100           0.0%   MX
4     consultas.tipoCredito                              100           0.0%   M|F|Q|TC|OT|NC|PP|LC
5     consultas.importeCredito                           100           0.0%   1000|0|7000|2000|6000|200
6     consultas.nombreOtorgante                          100           0.0%   CAMELINAS MECANICAS|SERVICI...
7     domicilios.direccion                               100           0.0%   AV MIGUEL 

## 10. 提取所有特征并保存为 CSV（可选）

In [26]:
# 提取所有特征并保存为 CSV
from datetime import datetime

print("=" * 70)
print("提取所有特征")
print("=" * 70)

# 收集所有特征键（从所有数据中）
all_feature_keys = set()
result_rows = []

print(f"\n正在处理 {len(df)} 条数据...")

for idx, row in df.iterrows():
    if idx % 1000 == 0:
        print(f"  进度: {idx}/{len(df)}")
    
    apply_id = row['apply_id']
    apply_time = row['apply_time']
    response_body = row['response_body']
    
    result_row = {
        'apply_id': apply_id,
        'apply_time': apply_time
    }
    
    if pd.isna(response_body) or response_body is None or response_body == '':
        result_rows.append(result_row)
        continue
    
    try:
        if isinstance(response_body, str):
            json_obj = json.loads(response_body)
        else:
            json_obj = response_body
        
        flattened = flatten_dict(json_obj)
        all_feature_keys.update(flattened.keys())
        result_row.update(flattened)
    
    except json.JSONDecodeError:
        pass
    except Exception:
        pass
    
    result_rows.append(result_row)

print(f"\n✅ 解析完成，共发现 {len(all_feature_keys)} 个特征字段")

# 确保所有行都有所有特征列
for row in result_rows:
    for key in all_feature_keys:
        if key not in row:
            row[key] = None

# 创建 DataFrame（保持列顺序：apply_id, apply_time, 然后按特征名排序）
feature_keys_sorted = sorted(all_feature_keys)
columns = ['apply_id', 'apply_time'] + feature_keys_sorted
result_df = pd.DataFrame(result_rows, columns=columns)

# 保存 CSV
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
output_feature_csv = f"{os.path.splitext(output_csv_path)[0]}_features_{current_time}.csv"

result_df.to_csv(output_feature_csv, index=False, encoding='utf-8-sig')
file_size = os.path.getsize(output_feature_csv)

print(f"\n✅ 特征提取完成！")
print(f"📁 输出文件: {output_feature_csv}")
print(f"📊 数据形状: {result_df.shape[0]:,} 行 × {result_df.shape[1]} 列")
print(f"💾 文件大小: {file_size / 1024 / 1024:.2f} MB")
print(f"\n前5列: {list(result_df.columns[:5])}")
print(f"特征列数: {len(feature_keys_sorted)}")

提取所有特征

正在处理 12546 条数据...
  进度: 0/12546
  进度: 1000/12546
  进度: 4000/12546
  进度: 19000/12546
  进度: 24000/12546
  进度: 30000/12546
  进度: 43000/12546
  进度: 44000/12546
  进度: 57000/12546
  进度: 60000/12546
  进度: 63000/12546
  进度: 75000/12546
  进度: 80000/12546
  进度: 81000/12546
  进度: 83000/12546

✅ 解析完成，共发现 85 个特征字段

✅ 特征提取完成！
📁 输出文件: /Users/zhanglifeng12703/Documents/OverseasPython/Mytest/cdc_pickle_pass_fpd7_features_20260123_101741.csv
📊 数据形状: 12,546 行 × 87 列
💾 文件大小: 66.10 MB

前5列: ['apply_id', 'apply_time', 'claveOtorgante', 'consultas.claveUnidadMonetaria', 'consultas.fechaConsulta']
特征列数: 85


## 4. 加载PKL文件

In [20]:
# 加载PKL文件
with open(pkl_file_path, 'rb') as f:
    data = pickle.load(f)

print(f"✅ 加载成功 | 类型: {type(data).__name__}")
if hasattr(data, '__len__'):
    print(f"   长度: {len(data):,}")
if isinstance(data, pd.DataFrame):
    print(f"   形状: {data.shape}")

✅ 加载成功 | 类型: DataFrame
   长度: 12,546
   形状: (12546, 11)


## 5. 分析数据结构

In [21]:
# 分析数据结构
print("=" * 70)
print("数据结构分析")
print("=" * 70)

if isinstance(data, pd.DataFrame):
    print(f"\n📊 数据类型: DataFrame")
    print(f"📏 形状: {data.shape[0]:,} 行 × {data.shape[1]} 列")
    print(f"\n📋 列名 ({len(data.columns)} 列):")
    for i, col in enumerate(data.columns, 1):
        dtype = str(data[col].dtype)
        null_count = data[col].isnull().sum()
        null_pct = (null_count / len(data)) * 100
        print(f"  {i:2d}. {col:30s} | 类型: {dtype:10s} | 缺失值: {null_count:5,} ({null_pct:5.1f}%)")
    
    print(f"\n📈 数据预览 (前5行):")
    print("-" * 70)
    print(data.head().to_string())
    
elif isinstance(data, dict):
    print(f"\n📊 数据类型: 字典")
    print(f"📏 键数量: {len(data)}")
    print(f"\n📋 键信息:")
    for i, (key, value) in enumerate(list(data.items())[:10], 1):
        print(f"  {i:2d}. {key}: {type(value).__name__}", end="")
        if isinstance(value, pd.DataFrame):
            print(f" | 形状: {value.shape}")
        elif isinstance(value, (list, tuple)):
            print(f" | 长度: {len(value)}")
        else:
            print()
    
elif isinstance(data, list):
    print(f"\n📊 数据类型: 列表")
    print(f"📏 元素数量: {len(data):,}")
    if len(data) > 0:
        print(f"📋 第一个元素类型: {type(data[0]).__name__}")
        if isinstance(data[0], dict):
            print(f"   键: {list(data[0].keys())[:10]}{'...' if len(data[0]) > 10 else ''}")
        print(f"\n📈 前3个元素预览:")
        print("-" * 70)
        pprint(data[:3], width=100, depth=2)
else:
    print(f"\n📊 数据类型: {type(data).__name__}")
    print(f"\n📈 内容预览:")
    print("-" * 70)
    pprint(data, width=100, depth=2)

print("\n" + "=" * 70)

数据结构分析

📊 数据类型: DataFrame
📏 形状: 12,546 行 × 11 列

📋 列名 (11 列):
   1. apply_id                       | 类型: object     | 缺失值:     0 (  0.0%)
   2. response_body                  | 类型: object     | 缺失值:     0 (  0.0%)
   3. apply_time                     | 类型: object     | 缺失值:     0 (  0.0%)
   4. approve_state                  | 类型: object     | 缺失值:     0 (  0.0%)
   5. credit_limit_amount            | 类型: object     | 缺失值:     0 (  0.0%)
   6. use_amount                     | 类型: object     | 缺失值:     0 (  0.0%)
   7. principal_amount_borrowed      | 类型: object     | 缺失值:     0 (  0.0%)
   8. fpd7                           | 类型: int32      | 缺失值:     0 (  0.0%)
   9. spd7                           | 类型: int32      | 缺失值:     0 (  0.0%)
  10. credit_apply_cnt               | 类型: int64      | 缺失值:     0 (  0.0%)
  11. blind_lend                     | 类型: float64    | 缺失值: 12,204 ( 97.3%)

📈 数据预览 (前5行):
----------------------------------------------------------------------
              a

## 6. 转换为DataFrame

In [16]:
# 转换为DataFrame
if isinstance(data, pd.DataFrame):
    df = data.copy()
elif isinstance(data, dict):
    dfs = [v.copy() for k, v in data.items() if isinstance(v, pd.DataFrame)]
    if len(dfs) == 1:
        df = dfs[0]
    elif len(dfs) > 1:
        for i, d in enumerate(dfs):
            d.insert(0, '_source_key', list(data.keys())[i])
        df = pd.concat(dfs, ignore_index=True)
    else:
        df = pd.DataFrame([data])
elif isinstance(data, list):
    df = pd.DataFrame(data)
else:
    df = pd.DataFrame([data])

print(f"DataFrame: {df.shape} | 列: {len(df.columns)}")
print(f"列名: {list(df.columns)[:20]}{'...' if len(df.columns) > 20 else ''}")

DataFrame: (12546, 11) | 列: 11
列名: ['apply_id', 'response_body', 'apply_time', 'approve_state', 'credit_limit_amount', 'use_amount', 'principal_amount_borrowed', 'fpd7', 'spd7', 'credit_apply_cnt', 'blind_lend']


## 7. 预览数据

In [17]:
print(df.head(10))
print(f"\n数据类型:")
print(df.dtypes)
print(f"\n缺失值: {df.isnull().sum().sum()}")
if df.select_dtypes(include=[np.number]).shape[1] > 0:
    print(f"\n统计信息:")
    print(df.describe())

               apply_id                                      response_body  \
0   1065991091661283329  {"claveOtorgante":"004676","consultas":[{"clav...   
1   1066560157648134145  {"claveOtorgante":"004676","consultas":[{"clav...   
2   1066719243236777985  {"claveOtorgante":"004676","consultas":[{"clav...   
3   1067025354145890305  {"claveOtorgante":"004676","consultas":[{"clav...   
8   1067575625355862017  {"claveOtorgante":"004676","consultas":[{"clav...   
10  1067977565575274497  {"claveOtorgante":"004676","consultas":[{"clav...   
13  1069974381770530819  {"claveOtorgante":"004676","consultas":[{"clav...   
16  1070209677258891265  {"claveOtorgante":"004676","consultas":[{"clav...   
17  1070514448138215425  {"claveOtorgante":"004676","consultas":[{"clav...   
21  1070828324348198913  {"claveOtorgante":"004676","consultas":[{"clav...   

                 apply_time approve_state credit_limit_amount     use_amount  \
0   2025-11-25 01:53:44.943    CYCLE_PASS        800.00000000

## 8. 保存CSV

In [10]:
df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
file_size = os.path.getsize(output_csv_path)
print(f"✅ 保存成功: {output_csv_path}")
print(f"   大小: {file_size / 1024 / 1024:.2f} MB | 行数: {len(df):,} | 列数: {len(df.columns)}")

✅ 保存成功: /Users/zhanglifeng12703/Documents/OverseasPython/Mytest/cdc_pickle_pass_fpd7.csv
   大小: 573.42 MB | 行数: 12,546 | 列数: 11


## 完成

输出文件: `cdc_pickle_pass_fpd7.csv`